# 1. Data Prep
Downloads WLASL and keeps only a quarter of the sign classes (glosses).

Run cells top to bottom. Output: `wlasl_quarter.csv`.

In [ ]:
!pip install -q kagglehub

In [ ]:
import json
import os

import kagglehub
import pandas as pd

## Download WLASL metadata + videos

In [ ]:
path = kagglehub.dataset_download("risangbaskoro/wlasl-processed")
print("Dataset downloaded to:", path)

## Load metadata into a DataFrame

In [ ]:
def load_wlasl_metadata(json_path: str) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        wlasl_data = json.load(f)

    records = []
    for entry in wlasl_data:
        gloss = entry.get("gloss")
        for inst in entry.get("instances", []):
            records.append({
                "gloss": gloss,
                "video_id": inst.get("video_id"),
                "fps": inst.get("fps"),
                "split": inst.get("split"),
                "url": inst.get("url"),
            })
    return pd.DataFrame(records)

json_file_path = os.path.join(path, "WLASL_v0.3.json")
df = load_wlasl_metadata(json_file_path)
print(f"Full dataset: {len(df)} instances across {df['gloss'].nunique()} classes")
df.head()

## Keep only a quarter of the classes
`mode="top"` keeps the 25% of glosses with the most video instances (more training data per class). `mode="random"` picks a random 25% of the classes instead.

In [ ]:
FRACTION = 0.25
MODE = "top"  # "top" or "random"

def select_quarter_classes(df: pd.DataFrame, fraction: float = 0.25, mode: str = "top") -> pd.DataFrame:
    counts = df["gloss"].value_counts()
    n_keep = max(1, int(len(counts) * fraction))

    if mode == "top":
        keep_glosses = counts.head(n_keep).index.tolist()
    elif mode == "random":
        keep_glosses = counts.sample(n=n_keep, random_state=42).index.tolist()
    else:
        raise ValueError("mode must be 'top' or 'random'")

    return df[df["gloss"].isin(keep_glosses)].reset_index(drop=True)

df_quarter = select_quarter_classes(df, fraction=FRACTION, mode=MODE)
print(f"Kept {df_quarter['gloss'].nunique()} classes ({MODE}, fraction={FRACTION}) -> {len(df_quarter)} instances")

## Map to local video files and keep only ones that exist on disk

In [ ]:
videos_dir = os.path.join(path, "videos")
df_quarter["local_path"] = df_quarter["video_id"].astype(str).apply(
    lambda vid: os.path.join(videos_dir, f"{vid}.mp4")
)
df_quarter["file_exists"] = df_quarter["local_path"].apply(os.path.exists)

df_final = df_quarter[df_quarter["file_exists"]].reset_index(drop=True)
print(f"Videos actually present on disk: {len(df_final)}")
df_final.head()

## Save the subset

In [ ]:
OUT_CSV = "wlasl_quarter.csv"
df_final.to_csv(OUT_CSV, index=False)
print(f"Saved {OUT_CSV}")